In [2]:
import pandas as pd
import sqlite3
import numpy as np
 
db_path = "amazon_analysis.db"
conn = sqlite3.connect(db_path)
 
print("Connected to database")

Connected to database


In [3]:
# Find products with high claimed discount but stable price
# Logic: If discount > 30% BUT price barely moved, MRP was probably inflated
 
fake_discounts = pd.read_sql("""
    SELECT 
        p.name,
        p.brand,
        p.category,
        COUNT(*) as days_tracked,
        MIN(ph.price) as min_price,
        MAX(ph.price) as max_price,
        MAX(ph.price) - MIN(ph.price) as price_range,
        ROUND(MAX(ph.discount_pct), 1) as max_discount_claimed,
        ROUND(AVG(ph.discount_pct), 1) as avg_discount
    FROM price_history ph
    JOIN products p ON ph.product_id = p.product_id
    GROUP BY p.product_id
    HAVING 
        MAX(ph.discount_pct) > 30 
        AND (MAX(ph.price) - MIN(ph.price)) < 500
    ORDER BY max_discount_claimed DESC
    LIMIT 20
""", conn)
 
print("FAKE DISCOUNT DETECTION (Top 20)")
print("="*80)
print(f"Products with suspicious pricing: {len(fake_discounts)}")
print("\nTop suspects (high discount claimed, but price barely moved):")
print(fake_discounts.to_string(index=False))

FAKE DISCOUNT DETECTION (Top 20)
Products with suspicious pricing: 20

Top suspects (high discount claimed, but price barely moved):
                                                                                                                                                                                                 name       brand        category  days_tracked  min_price  max_price  price_range  max_discount_claimed  avg_discount
                   Milton Turbo Yogurt Maker Machine | Automatic with One-Touch Operation | Electric Yogurt Maker | Make Thick, Creamy Curd & Yogurt at Home | Energy Efficient | Easy to Use | White      Milton home_appliances            14      699.0      998.0        299.0                  99.0          44.8
Smart Foldable Electric Kettle – Quick Boil Kettle for Hot Water | Instant Tea & Coffee Maker, 1 Liter | Food-Grade Silicone, 304 Stainless Steel Base, Detachable Power Cord | Home, Office & Travel       Smart home_appliances             3      

In [4]:
price_volatility = pd.read_sql("""
    SELECT 
        p.name,
        p.brand,
        p.category,
        COUNT(*) as snapshots,
        ROUND(MIN(ph.price), 0) as min_price,
        ROUND(MAX(ph.price), 0) as max_price,
        ROUND(MAX(ph.price) - MIN(ph.price), 0) as price_range,
        ROUND(AVG(ph.price), 0) as avg_price
    FROM price_history ph
    JOIN products p ON ph.product_id = p.product_id
    GROUP BY p.product_id
    HAVING COUNT(*) >= 3
    ORDER BY price_range DESC
    LIMIT 20
""", conn)
 
print("\nPRICE VOLATILITY ANALYSIS (Top 20 most volatile products)")
print("="*80)
print(price_volatility.to_string(index=False))


PRICE VOLATILITY ANALYSIS (Top 20 most volatile products)
                                                                                                                                                                                                    name     brand    category  snapshots  min_price  max_price  price_range  avg_price
          Alienware Area-15 Gaming Laptop,Intel Core Ultra 9 275HX+24GB RTX 5090,64GB,2TB SSD with Logitech G502 LS Wireless Gaming Mouse, 25000DPI Programmable and Logitech GProX Wired Gaming Headset Alienware     laptops         20   417024.0   528280.0     111256.0   513496.0
 ASUS ROG Strix G16,Intel Core Ultra 9 275HX,Gaming Laptop(RTX 5070-8GB/115W TGP/32GB/1TB /2.5K QHD+/16"/240Hz/90WHrs/Windows 11/M365 Basic(1Year)*/Office Home 2024/Eclipse Gray/2.65 Kg)G615LP-S5022WS      Asus     laptops         20   213990.0   295399.0      81409.0   250810.0
   Lenovo Thinkpad P16s Intel Core Ultra 7 155H 16"(40.64 cm) WUXGA IPS 300Nits, AI Ready Thin and Li

In [5]:
# Products with highest review growth = highest demand signal
 
review_growth = pd.read_sql("""
    SELECT 
        p.name,
        p.brand,
        p.category,
        ms1.reviews as day1_reviews,
        ms5.reviews as day5_reviews,
        CAST(ms5.reviews - ms1.reviews AS INTEGER) as review_growth,
        ROUND(CAST(ms5.reviews - ms1.reviews AS FLOAT) / 4, 1) as reviews_per_day,
        ROUND(ms1.rating, 1) as rating_start,
        ROUND(ms5.rating, 1) as rating_end
    FROM market_signals ms1
    JOIN market_signals ms5 ON ms1.product_id = ms5.product_id
    JOIN products p ON ms1.product_id = p.product_id
    WHERE ms1.date = (SELECT MIN(date) FROM market_signals)
      AND ms5.date = (SELECT MAX(date) FROM market_signals)
      AND ms1.reviews > 0
    ORDER BY review_growth DESC
    LIMIT 20
""", conn)
 
print("\nREVIEW GROWTH ANALYSIS (Demand signals - Top 20)")
print("="*80)
print(f"Products with growth data: {len(review_growth)}")
print("\nTop products by review growth (5-day demand):")
print(review_growth.to_string(index=False))


REVIEW GROWTH ANALYSIS (Demand signals - Top 20)
Products with growth data: 20

Top products by review growth (5-day demand):
                                                                                                                                                                                                    name     brand        category  day1_reviews  day5_reviews  review_growth  reviews_per_day  rating_start  rating_end
                                                                     Bajaj Military Series Rex 750W 4 Jar Mixer Grinder | DuraCut Blades | 2-In-1 Function Blade In Dry Jar | 2 Yrs Warranty 【Red/Black】     Bajaj home_appliances         539.0        5825.0           5286           1321.5           3.9         4.0
                      Milton Turbo Yogurt Maker Machine | Automatic with One-Touch Operation | Electric Yogurt Maker | Make Thick, Creamy Curd & Yogurt at Home | Energy Efficient | Easy to Use | White    Milton home_appliances          16.0       

In [6]:
# Understanding discount strategy by brand
 
brand_discount = pd.read_sql("""
    SELECT 
        p.brand,
        p.category,
        COUNT(DISTINCT p.product_id) as num_products,
        ROUND(AVG(ph.discount_pct), 1) as avg_discount,
        ROUND(MIN(ph.discount_pct), 1) as min_discount,
        ROUND(MAX(ph.discount_pct), 1) as max_discount
    FROM price_history ph
    JOIN products p ON ph.product_id = p.product_id
    GROUP BY p.brand, p.category
    HAVING COUNT(DISTINCT p.product_id) >= 3
    ORDER BY avg_discount DESC
    LIMIT 30
""", conn)
 
print("\nAVERAGE DISCOUNT BY BRAND & CATEGORY")
print("="*80)
print(brand_discount.to_string(index=False))


AVERAGE DISCOUNT BY BRAND & CATEGORY
         brand        category  num_products  avg_discount  min_discount  max_discount
            Ro home_appliances             3          77.8          74.0          81.0
       Digital home_appliances             4          74.8          54.0          91.0
            Dr home_appliances             6          73.9          73.0          78.0
     Butterfly home_appliances             3          71.4          37.0          80.0
     Certified         laptops            10          70.8          53.0          81.0
         Faber home_appliances             3          70.4          59.0          76.0
          Xech home_appliances             5          69.0          67.0          72.0
         Qlect home_appliances             5          68.0          65.0          88.0
        Seznik home_appliances             4          67.8          26.0          74.0
      Nutripro home_appliances             3          67.6          57.0          70.0
     

In [7]:
# For each category, find products with price changes to estimate elasticity
 
elasticity_data = pd.read_sql("""
    SELECT 
        p.category,
        p.product_id,
        p.name,
        p.brand,
        ph1.price as price_start,
        ph5.price as price_end,
        ROUND(CAST(ph5.price - ph1.price AS FLOAT) / ph1.price * 100, 1) as price_change_pct,
        ms1.reviews as reviews_start,
        ms5.reviews as reviews_end,
        ROUND(CAST(ms5.reviews - ms1.reviews AS FLOAT) / ms1.reviews * 100, 1) as review_change_pct
    FROM products p
    JOIN price_history ph1 ON p.product_id = ph1.product_id
    JOIN price_history ph5 ON p.product_id = ph5.product_id
    JOIN market_signals ms1 ON p.product_id = ms1.product_id
    JOIN market_signals ms5 ON p.product_id = ms5.product_id
    WHERE ph1.date = (SELECT MIN(date) FROM price_history)
      AND ph5.date = (SELECT MAX(date) FROM price_history)
      AND ms1.date = (SELECT MIN(date) FROM market_signals)
      AND ms5.date = (SELECT MAX(date) FROM market_signals)
      AND ms1.reviews > 0
      AND ABS(ph5.price - ph1.price) > 0
    ORDER BY p.category, ABS(price_change_pct) DESC
""", conn)
 
print("\nPRICE-DEMAND CORRELATION (Products with price changes)")
print("="*80)
print(f"Products with measurable price changes: {len(elasticity_data)}")
print("\nShowing top 20 by price change magnitude:")
print(elasticity_data.head(20).to_string(index=False))


PRICE-DEMAND CORRELATION (Products with price changes)
Products with measurable price changes: 183

Showing top 20 by price change magnitude:
       category  product_id                                                                                                                                                                                                    name      brand  price_start  price_end  price_change_pct  reviews_start  reviews_end  review_change_pct
home_appliances         609                                                 GANESH Plastic Countertop Accessories Items Tiered Shelf Masala Box For, Containers Set, Spice Box For, Spice Rack Set For Storage, Container Set Of 12     Ganesh        779.0      584.0             -25.0          484.0        488.0                0.8
home_appliances         563                            QOKZEK Non-Scratch Dish Wash Cloth | Versatile Wire Scrubber for Kitchen Cleaning | Reusable Mesh Cloth for Dishes, Sinks, and Home Appliances | E

In [8]:
# High-level overview per category
 
category_summary = pd.read_sql("""
    SELECT 
        p.category,
        COUNT(DISTINCT p.product_id) as num_products,
        COUNT(DISTINCT p.brand) as num_brands,
        ROUND(AVG(ph.price), 0) as avg_price,
        ROUND(AVG(ph.mrp), 0) as avg_mrp,
        ROUND(AVG(ph.discount_pct), 1) as avg_discount,
        ROUND(AVG(ms.reviews), 0) as avg_reviews,
        ROUND(AVG(ms.rating), 2) as avg_rating,
        ROUND(CAST(SUM(ms.in_stock) AS FLOAT) / COUNT(*) * 100, 1) as pct_in_stock
    FROM products p
    JOIN price_history ph ON p.product_id = ph.product_id
    JOIN market_signals ms ON p.product_id = ms.product_id
    GROUP BY p.category
    ORDER BY avg_discount DESC
""", conn)
 
print("\nCATEGORY SUMMARY STATISTICS")
print("="*80)
print(category_summary.to_string(index=False))


CATEGORY SUMMARY STATISTICS
       category  num_products  num_brands  avg_price  avg_mrp  avg_discount  avg_reviews  avg_rating  pct_in_stock
home_appliances           822         338     1958.0  16459.0          48.6       9133.0        4.04         100.0
        laptops           621          24    93139.0 116456.0          20.8        143.0        4.02         100.0
    smartphones           715          51    32453.0  38080.0          14.0        659.0        4.14         100.0


In [9]:
# Estimate elasticity: for each category, what's the price-demand correlation?
 
elasticity_by_category = elasticity_data.dropna()
 
print("\nELASTICITY ESTIMATION BY CATEGORY")
print("="*80)
 
for category in elasticity_by_category['category'].unique():
    cat_data = elasticity_by_category[elasticity_by_category['category'] == category]
    
    if len(cat_data) > 5:  # Need at least 5 products with price changes
        # Simple correlation: average price change vs average review change
        avg_price_change = cat_data['price_change_pct'].mean()
        avg_review_change = cat_data['review_change_pct'].mean()
        
        # Elasticity = % change in demand / % change in price
        if avg_price_change != 0:
            elasticity = avg_review_change / avg_price_change
        else:
            elasticity = 0
        
        products_with_changes = len(cat_data)
        
        print(f"\n{category.upper()}")
        print(f"  Products with price changes: {products_with_changes}")
        print(f"  Avg price change: {avg_price_change:.1f}%")
        print(f"  Avg review change: {avg_review_change:.1f}%")
        print(f"  Elasticity coefficient: {elasticity:.2f}")
        
        if elasticity > 0:
            elasticity_pct = abs(elasticity * 10)
            print(f"  → Interpretation: 10% price drop = {elasticity_pct:.1f}% demand increase")
    else:
        print(f"\n{category.upper()}: Not enough products with price changes ({len(cat_data)})")


ELASTICITY ESTIMATION BY CATEGORY

HOME_APPLIANCES
  Products with price changes: 46
  Avg price change: 0.2%
  Avg review change: 32.3%
  Elasticity coefficient: 176.93
  → Interpretation: 10% price drop = 1769.3% demand increase

LAPTOPS
  Products with price changes: 83
  Avg price change: 2.7%
  Avg review change: 30.1%
  Elasticity coefficient: 11.00
  → Interpretation: 10% price drop = 110.0% demand increase

SMARTPHONES
  Products with price changes: 54
  Avg price change: 4.4%
  Avg review change: 31.6%
  Elasticity coefficient: 7.21
  → Interpretation: 10% price drop = 72.1% demand increase


In [10]:
# ELASTICITY FIX: Only products with >5% price drops
elastic_data = elasticity_data[elasticity_data['price_change_pct'] < -5].copy()

print("\nELASTICITY (CORRECTED - Only products with price drops > 5%)")
print("="*80)

for category in elastic_data['category'].unique():
    cat_data = elastic_data[elastic_data['category'] == category]
    
    if len(cat_data) > 5:
        avg_price_change = cat_data['price_change_pct'].mean()
        avg_review_change = cat_data['review_change_pct'].mean()
        
        elasticity = avg_review_change / abs(avg_price_change)
        
        print(f"\n{category.upper()}")
        print(f"  Products with >5% price drop: {len(cat_data)}")
        print(f"  Avg price drop: {abs(avg_price_change):.1f}%")
        print(f"  Avg review change: {avg_review_change:.1f}%")
        print(f"  Elasticity coefficient: {elasticity:.2f}")
        print(f"  → Interpretation: 10% price drop = {elasticity * 10:.1f}% demand increase")
    else:
        print(f"\n{category.upper()}: Not enough products with >5% price drop ({len(cat_data)})")


ELASTICITY (CORRECTED - Only products with price drops > 5%)

HOME_APPLIANCES
  Products with >5% price drop: 10
  Avg price drop: 14.4%
  Avg review change: 11.2%
  Elasticity coefficient: 0.77
  → Interpretation: 10% price drop = 7.7% demand increase

LAPTOPS
  Products with >5% price drop: 6
  Avg price drop: 6.9%
  Avg review change: 1.8%
  Elasticity coefficient: 0.27
  → Interpretation: 10% price drop = 2.7% demand increase

SMARTPHONES: Not enough products with >5% price drop (4)


In [11]:
# Print key findings
print("\n" + "="*80)
print("KEY INSIGHTS FROM 5-DAY DATA")
print("="*80)
 
# Insight 1: Fake discounts
fake_count = len(fake_discounts)
print(f"\n1. FAKE DISCOUNTS:")
print(f"   Found {fake_count} products with suspicious pricing")
print(f"   (High discount claimed but price barely moved)")
if fake_count > 0:
    print(f"   Example: {fake_discounts.iloc[0]['name']} - {fake_discounts.iloc[0]['max_discount_claimed']}% discount but only ₹{fake_discounts.iloc[0]['price_range']} price move")
 
# Insight 2: Most volatile products
most_volatile = price_volatility.iloc[0] if len(price_volatility) > 0 else None
if most_volatile is not None:
    print(f"\n2. PRICE VOLATILITY:")
    print(f"   Most volatile: {most_volatile['name']}")
    print(f"   Price range: ₹{most_volatile['min_price']} to ₹{most_volatile['max_price']} (₹{most_volatile['price_range']} range)")
 
# Insight 3: Highest demand
top_demand = review_growth.iloc[0] if len(review_growth) > 0 else None
if top_demand is not None:
    print(f"\n3. HIGHEST DEMAND (Review Growth):")
    print(f"   Product: {top_demand['name']}")
    print(f"   Reviews grew from {int(top_demand['day1_reviews'])} to {int(top_demand['day5_reviews'])} (+{int(top_demand['review_growth'])} reviews)")
    print(f"   Growth rate: {top_demand['reviews_per_day']} reviews/day")
 
# Insight 4: Category comparison
print(f"\n4. DISCOUNT STRATEGY BY CATEGORY:")
for idx, row in category_summary.iterrows():
    print(f"   {row['category']}: {row['avg_discount']}% avg discount, {int(row['avg_price'])} avg price")
 
# Insight 5: Elasticity preview
elasticity_summary = elasticity_data.groupby('category').apply(
    lambda x: {
        'category': x['category'].iloc[0],
        'products_with_changes': len(x),
        'avg_price_change': x['price_change_pct'].mean(),
        'avg_review_change': x['review_change_pct'].mean()
    }
)
 
print(f"\n5. ELASTICITY PREVIEW (5-day data):")
for cat in elasticity_summary:
    print(f"   {cat['category']}: Avg price change {cat['avg_price_change']:.1f}%, Review change {cat['avg_review_change']:.1f}%")
 
print("\n Analysis complete!")
conn.close()


KEY INSIGHTS FROM 5-DAY DATA

1. FAKE DISCOUNTS:
   Found 20 products with suspicious pricing
   (High discount claimed but price barely moved)
   Example: Milton Turbo Yogurt Maker Machine | Automatic with One-Touch Operation | Electric Yogurt Maker | Make Thick, Creamy Curd & Yogurt at Home | Energy Efficient | Easy to Use | White - 99.0% discount but only ₹299.0 price move

2. PRICE VOLATILITY:
   Most volatile: Alienware Area-15 Gaming Laptop,Intel Core Ultra 9 275HX+24GB RTX 5090,64GB,2TB SSD with Logitech G502 LS Wireless Gaming Mouse, 25000DPI Programmable and Logitech GProX Wired Gaming Headset
   Price range: ₹417024.0 to ₹528280.0 (₹111256.0 range)

3. HIGHEST DEMAND (Review Growth):
   Product: Bajaj Military Series Rex 750W 4 Jar Mixer Grinder | DuraCut Blades | 2-In-1 Function Blade In Dry Jar | 2 Yrs Warranty 【Red/Black】
   Reviews grew from 539 to 5825 (+5286 reviews)
   Growth rate: 1321.5 reviews/day

4. DISCOUNT STRATEGY BY CATEGORY:
   home_appliances: 48.6% avg dis

C:\Users\Ashish\AppData\Local\Temp\ipykernel_16028\4142662475.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  elasticity_summary = elasticity_data.groupby('category').apply(
